# Vista Consolidada ML Churn para Power BI

## Objetivo
Crear una vista única (`vista_ml_churn_powerbi`) que consolide:
- Features del modelo ML (18 variables)
- Scoring de churn (probabilidad_fuga)
- Segmentaciones de negocio (riesgo, valor, prioridad)
- Datos de comportamiento (deportes, transacciones, última apuesta)
- Recomendaciones de acción

## Estructura para 5 Dashboards en Power BI
1. **Resumen Ejecutivo**: KPIs principales, Top N usuarios
2. **Recencia/Frecuencia**: Análisis temporal (días inactividad, tendencias)
3. **Preferencias**: Deportes favoritos, mix transacciones
4. **Valor y Priorización**: Segmentación por valor, matriz riesgo/valor
5. **Campañas y Acciones**: Recomendaciones personalizadas

## Conexión desde Power BI
1. En Power BI Desktop: **Obtener datos > Base de datos > Databricks**
2. Host: `dbc-cbc74f79-006a.cloud.databricks.com`
3. Ruta HTTP: `/sql/1.0/warehouses/<warehouse_id>`
4. Tabla: `workspace.apuestas_db.vista_ml_churn_powerbi`
5. Modo: **DirectQuery** (recomendado) o Import

In [0]:
%sql
-- ============================================================================
-- VISTA CONSOLIDADA PARA POWER BI - PREDICCIÓN DE CHURN
-- Organizada para 5 dashboards: Resumen, Recencia/Frecuencia, Preferencias,
-- Valor, y Campañas
-- ============================================================================

CREATE OR REPLACE VIEW workspace.apuestas_db.vista_ml_churn_powerbi AS

-- CTEs para cálculos auxiliares
WITH ultima_apuesta_detalle AS (
    SELECT 
        a.id_usuario,
        a.monto AS ultima_apuesta_monto,
        a.cuota AS ultima_apuesta_cuota,
        a.fecha_apuesta AS ultima_apuesta_fecha,
        e.tipo_deporte AS ultima_apuesta_deporte,
        ea.nombre_estado AS ultima_apuesta_estado
    FROM workspace.apuestas_db.apuesta a
    INNER JOIN workspace.apuestas_db.evento e ON a.id_evento = e.id_evento
    INNER JOIN workspace.apuestas_db.estado_apuesta ea ON a.id_estado_apuesta = ea.id_estado
    QUALIFY ROW_NUMBER() OVER (PARTITION BY a.id_usuario ORDER BY a.fecha_apuesta DESC) = 1
),

deporte_favorito_base AS (
    SELECT 
        a.id_usuario,
        e.tipo_deporte,
        COUNT(*) AS n_apuestas_deporte
    FROM workspace.apuestas_db.apuesta a
    INNER JOIN workspace.apuestas_db.evento e ON a.id_evento = e.id_evento
    GROUP BY a.id_usuario, e.tipo_deporte
),

deporte_favorito AS (
    SELECT 
        id_usuario,
        tipo_deporte AS deporte_favorito,
        n_apuestas_deporte AS veces_apostado_favorito,
        ROUND(n_apuestas_deporte * 100.0 / SUM(n_apuestas_deporte) OVER (PARTITION BY id_usuario), 2) AS pct_apuestas_favorito
    FROM deporte_favorito_base
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id_usuario ORDER BY n_apuestas_deporte DESC) = 1
),

apuestas_pendientes AS (
    SELECT 
        id_usuario,
        COUNT(*) AS n_apuestas_pendientes,
        SUM(monto) AS monto_apuestas_pendientes
    FROM workspace.apuestas_db.apuesta
    WHERE id_estado_apuesta = 1
    GROUP BY id_usuario
),

mix_transacciones AS (
    SELECT 
        id_usuario,
        SUM(CASE WHEN tipo_transaccion = 'deposito' THEN 1 ELSE 0 END) AS n_transacciones_deposito,
        SUM(CASE WHEN tipo_transaccion = 'retiro' THEN 1 ELSE 0 END) AS n_transacciones_retiro,
        SUM(CASE WHEN tipo_transaccion = 'deposito' THEN monto ELSE 0 END) AS monto_total_depositado_tx,
        SUM(CASE WHEN tipo_transaccion = 'retiro' THEN monto ELSE 0 END) AS monto_total_retirado,
        CASE 
            WHEN SUM(CASE WHEN tipo_transaccion IN ('deposito', 'retiro') THEN 1 ELSE 0 END) > 0 THEN
                ROUND(SUM(CASE WHEN tipo_transaccion = 'deposito' THEN 1 ELSE 0 END) * 100.0 / 
                      SUM(CASE WHEN tipo_transaccion IN ('deposito', 'retiro') THEN 1 ELSE 0 END), 2)
            ELSE 0
        END AS pct_transacciones_deposito
    FROM workspace.apuestas_db.transaccion
    GROUP BY id_usuario
),

desglose_apuestas AS (
    SELECT 
        id_usuario,
        SUM(CASE WHEN id_estado_apuesta = 2 THEN 1 ELSE 0 END) AS n_apuestas_ganadas,
        SUM(CASE WHEN id_estado_apuesta = 3 THEN 1 ELSE 0 END) AS n_apuestas_perdidas,
        SUM(CASE WHEN id_estado_apuesta = 1 THEN 1 ELSE 0 END) AS n_apuestas_pendientes_desglose
    FROM workspace.apuestas_db.apuesta
    GROUP BY id_usuario
)

SELECT 
    -- ========================================================================
    -- DASHBOARD 1: RESUMEN EJECUTIVO
    -- ========================================================================
    u.id_usuario,
    u.nombre,
    u.apellido,
    u.numero_documento,
    u.tipo_documento,
    u.fecha_registro,
    s.churn AS churn_real,
    s.probabilidad_fuga,
    CASE 
        WHEN s.probabilidad_fuga >= 0.80 THEN 'Crítico'
        WHEN s.probabilidad_fuga >= 0.60 THEN 'Alto Riesgo'
        WHEN s.probabilidad_fuga >= 0.40 THEN 'Riesgo Moderado'
        WHEN s.probabilidad_fuga >= 0.20 THEN 'Riesgo Bajo'
        ELSE 'Estable'
    END AS segmento_riesgo,
    m.saldo,
    COALESCE(df.deporte_favorito, 'Sin Apuestas') AS deporte_favorito,
    COALESCE(ap.n_apuestas_pendientes, 0) AS n_apuestas_pendientes,
    COALESCE(ap.monto_apuestas_pendientes, 0) AS monto_apuestas_pendientes,
    
    -- ========================================================================
    -- DASHBOARD 2: RECENCIA/FRECUENCIA
    -- ========================================================================
    m.dias_desde_ultima_apuesta,
    m.dias_desde_ultimo_deposito,
    m.antiguedad_dias,
    m.n_apuestas_total,
    m.n_apuestas_ult30d,
    m.n_apuestas_previas,
    m.tendencia_frecuencia,
    CASE 
        WHEN m.tendencia_frecuencia < -10 THEN 'Descendente'
        WHEN m.tendencia_frecuencia > 5 THEN 'Ascendente'
        ELSE 'Estable'
    END AS tendencia_actividad,
    CASE WHEN m.dias_desde_ultima_apuesta > 60 THEN 'Sí' ELSE 'No' END AS inactivo_60_dias,
    CASE WHEN m.dias_desde_ultimo_deposito > 90 THEN 'Sí' ELSE 'No' END AS sin_depositos_90_dias,
    
    -- ========================================================================
    -- DASHBOARD 3: PREFERENCIAS
    -- ========================================================================
    COALESCE(df.veces_apostado_favorito, 0) AS veces_apostado_favorito,
    COALESCE(df.pct_apuestas_favorito, 0) AS pct_apuestas_favorito,
    COALESCE(mt.n_transacciones_deposito, 0) AS n_transacciones_deposito,
    COALESCE(mt.n_transacciones_retiro, 0) AS n_transacciones_retiro,
    COALESCE(mt.monto_total_depositado_tx, 0) AS monto_total_depositado_tx,
    COALESCE(mt.monto_total_retirado, 0) AS monto_total_retirado,
    COALESCE(mt.pct_transacciones_deposito, 0) AS pct_transacciones_deposito,
    CASE 
        WHEN COALESCE(mt.pct_transacciones_deposito, 0) > 80 THEN 'Solo Deposita'
        WHEN COALESCE(mt.pct_transacciones_deposito, 0) > 50 THEN 'Más Deposita que Retira'
        WHEN COALESCE(mt.pct_transacciones_deposito, 0) >= 50 THEN 'Balanceado'
        ELSE 'Más Retira que Deposita'
    END AS patron_financiero,
    
    -- ========================================================================
    -- DASHBOARD 4: VALOR Y PRIORIZACIÓN
    -- ========================================================================
    m.monto_promedio_apostado,
    m.monto_promedio_ult30d,
    m.monto_total_depositado,
    m.n_depositos,
    CASE 
        WHEN m.monto_total_depositado > 15000 THEN 'VIP'
        WHEN m.monto_total_depositado > 8000 THEN 'Alto Valor'
        WHEN m.monto_total_depositado > 3000 THEN 'Medio Valor'
        ELSE 'Bajo Valor'
    END AS segmento_valor,
    COALESCE(da.n_apuestas_ganadas, 0) AS n_apuestas_ganadas,
    COALESCE(da.n_apuestas_perdidas, 0) AS n_apuestas_perdidas,
    COALESCE(da.n_apuestas_pendientes_desglose, 0) AS n_apuestas_pendientes_desglose,
    m.tasa_perdida,
    CASE 
        WHEN m.tasa_perdida >= 0.80 THEN 'Alta Frustración'
        WHEN m.tasa_perdida >= 0.60 THEN 'Frustración Moderada'
        ELSE 'Normal'
    END AS nivel_frustracion,
    CASE 
        WHEN s.probabilidad_fuga >= 0.80 AND m.monto_total_depositado > 15000 THEN 1
        WHEN s.probabilidad_fuga >= 0.80 AND m.monto_total_depositado > 8000 THEN 2
        WHEN s.probabilidad_fuga >= 0.80 THEN 3
        WHEN s.probabilidad_fuga >= 0.60 AND m.monto_total_depositado > 15000 THEN 4
        WHEN s.probabilidad_fuga >= 0.60 THEN 5
        ELSE 6
    END AS prioridad_contacto,
    
    -- ========================================================================
    -- DASHBOARD 5: CAMPAÑAS Y ACCIONES
    -- ========================================================================
    ua.ultima_apuesta_monto,
    ua.ultima_apuesta_cuota,
    ua.ultima_apuesta_fecha,
    ua.ultima_apuesta_deporte,
    ua.ultima_apuesta_estado,
    CASE 
        WHEN s.probabilidad_fuga >= 0.80 AND m.saldo > 10000 THEN 'Contacto Urgente + Bono Retención'
        WHEN s.probabilidad_fuga >= 0.80 THEN 'Contacto Urgente'
        WHEN s.probabilidad_fuga >= 0.60 AND m.n_apuestas_total > 15 THEN 'Campaña Reactivación VIP'
        WHEN s.probabilidad_fuga >= 0.60 THEN 'Campaña Reactivación'
        WHEN s.probabilidad_fuga >= 0.40 AND m.dias_desde_ultima_apuesta > 30 THEN 'Email Recordatorio'
        WHEN s.probabilidad_fuga >= 0.40 THEN 'Monitoreo Activo'
        ELSE 'Campaña Fidelización'
    END AS accion_recomendada,
    m.sin_apuestas_obs,
    m.sin_depositos_obs,
    m.sin_apuestas_resueltas

FROM workspace.apuestas_db.usuario u
INNER JOIN workspace.apuestas_db.scoring_churn_usuarios s ON u.id_usuario = s.id_usuario
INNER JOIN workspace.apuestas_db.tabla_maestra_churn m ON u.id_usuario = m.id_usuario
LEFT JOIN ultima_apuesta_detalle ua ON u.id_usuario = ua.id_usuario
LEFT JOIN deporte_favorito df ON u.id_usuario = df.id_usuario
LEFT JOIN apuestas_pendientes ap ON u.id_usuario = ap.id_usuario
LEFT JOIN mix_transacciones mt ON u.id_usuario = mt.id_usuario
LEFT JOIN desglose_apuestas da ON u.id_usuario = da.id_usuario;

# Diccionario de Campos de la Vista

## Dashboard 1: Resumen Ejecutivo
| Campo | Tipo | Descripción | Uso en Power BI |
|-------|------|-------------|----------------|
| id_usuario | INT | Identificador único | Filtros, relaciones |
| nombre, apellido | STRING | Datos personales | Tablas Top N |
| numero_documento, tipo_documento | STRING | Identificación | Tablas detalle |
| fecha_registro | DATE | Fecha de alta | Análisis temporal |
| churn_real | INT | Etiqueta real (0/1) | Validación modelo |
| probabilidad_fuga | DOUBLE | Probabilidad 0-1 | KPI principal, scatter |
| segmento_riesgo | STRING | 5 categorías | Filtro principal, KPIs |
| saldo | DECIMAL | Saldo actual | KPI, Top N |
| deporte_favorito | STRING | Deporte más apostado | Segmentación |
| n_apuestas_pendientes | INT | Apuestas sin resolver | Indicador retención |
| monto_apuestas_pendientes | DECIMAL | Dinero en juego | Indicador retención |

## Dashboard 2: Recencia/Frecuencia
| Campo | Tipo | Descripción | Uso en Power BI |
|-------|------|-------------|----------------|
| dias_desde_ultima_apuesta | INT | Días inactividad apuestas | Eje X scatter, histograma |
| dias_desde_ultimo_deposito | INT | Días inactividad depósitos | Eje X scatter |
| antiguedad_dias | INT | Días desde registro | Segmentación, filtro |
| n_apuestas_total | INT | Total histórico | Histograma, KPI |
| n_apuestas_ult30d | INT | Apuestas últimos 30 días | KPI actividad |
| n_apuestas_previas | INT | Apuestas antes de ventana obs | Feature ML |
| tendencia_frecuencia | INT | Cambio en frecuencia | Eje Y scatter |
| tendencia_actividad | STRING | 3 categorías | Filtro, visualización |
| inactivo_60_dias | STRING | Sí/No | Filtro, KPI |
| sin_depositos_90_dias | STRING | Sí/No | Filtro, KPI |

## Dashboard 3: Preferencias
| Campo | Tipo | Descripción | Uso en Power BI |
|-------|------|-------------|----------------|
| veces_apostado_favorito | INT | Apuestas en deporte favorito | Tabla detalle |
| pct_apuestas_favorito | DECIMAL | % apuestas en favorito | Indicador preferencia |
| n_transacciones_deposito | INT | Cantidad de depósitos | Histograma |
| n_transacciones_retiro | INT | Cantidad de retiros | Histograma |
| monto_total_depositado_tx | DECIMAL | Dinero depositado | KPI |
| monto_total_retirado | DECIMAL | Dinero retirado | KPI |
| pct_transacciones_deposito | DECIMAL | % depósitos vs retiros | Indicador comportamiento |
| patron_financiero | STRING | 4 categorías | Filtro, segmentación |

## Dashboard 4: Valor y Priorización
| Campo | Tipo | Descripción | Uso en Power BI |
|-------|------|-------------|----------------|
| monto_promedio_apostado | DECIMAL | Promedio por apuesta | KPI, boxplot |
| monto_promedio_ult30d | DECIMAL | Promedio últimos 30 días | Tendencia |
| monto_total_depositado | DECIMAL | Lifetime value | KPI principal |
| n_depositos | INT | Cantidad depósitos | Indicador compromiso |
| segmento_valor | STRING | 4 categorías (VIP...) | Filtro principal, matriz |
| n_apuestas_ganadas | INT | Apuestas ganadas | KPI, tasa éxito |
| n_apuestas_perdidas | INT | Apuestas perdidas | KPI, tasa éxito |
| n_apuestas_pendientes_desglose | INT | Pendientes (desglose) | KPI |
| tasa_perdida | DOUBLE | % apuestas perdidas | KPI frustración |
| nivel_frustracion | STRING | 3 categorías | Filtro, segmentación |
| prioridad_contacto | INT | 1-6 (1=urgente) | Ordenamiento tablas |

## Dashboard 5: Campañas y Acciones
| Campo | Tipo | Descripción | Uso en Power BI |
|-------|------|-------------|----------------|
| ultima_apuesta_monto | DECIMAL | Monto última apuesta | Contexto personalización |
| ultima_apuesta_cuota | DECIMAL | Cuota última apuesta | Contexto personalización |
| ultima_apuesta_fecha | TIMESTAMP | Fecha última apuesta | Cálculo recencia |
| ultima_apuesta_deporte | STRING | Deporte última apuesta | Personalización campaña |
| ultima_apuesta_estado | STRING | Estado última apuesta | Contexto abandono |
| accion_recomendada | STRING | 7 tipos de acción | Filtro principal, tabla |
| sin_apuestas_obs | INT | Flag binario | Feature técnica |
| sin_depositos_obs | INT | Flag binario | Feature técnica |
| sin_apuestas_resueltas | INT | Flag binario | Feature técnica |

## Campos Calculados Sugeridos en Power BI
```dax
// Tasa de éxito
Tasa_Exito = DIVIDE([n_apuestas_ganadas], [n_apuestas_ganadas] + [n_apuestas_perdidas], 0)

// ROI aproximado
ROI = DIVIDE([monto_total_retirado] - [monto_total_depositado_tx], [monto_total_depositado_tx], 0)

// Nivel de engagement
Engagement_Score = ([n_apuestas_ult30d] * 0.4) + ([n_transacciones_deposito] * 0.3) + (1 / ([dias_desde_ultima_apuesta] + 1) * 0.3)
```

In [0]:
%sql
-- Validar que la vista funciona correctamente
SELECT 
    COUNT(*) AS total_usuarios,
    COUNT(DISTINCT segmento_riesgo) AS segmentos_riesgo,
    COUNT(DISTINCT deporte_favorito) AS deportes_unicos,
    COUNT(DISTINCT segmento_valor) AS segmentos_valor,
    COUNT(DISTINCT accion_recomendada) AS acciones_unicas,
    SUM(CASE WHEN ultima_apuesta_deporte IS NOT NULL THEN 1 ELSE 0 END) AS usuarios_con_ultima_apuesta,
    ROUND(AVG(probabilidad_fuga), 4) AS prob_fuga_promedio
FROM workspace.apuestas_db.vista_ml_churn_powerbi;

total_usuarios,segmentos_riesgo,deportes_unicos,segmentos_valor,acciones_unicas,usuarios_con_ultima_apuesta,prob_fuga_promedio
800,5,5,4,5,800,0.3828


In [0]:
%sql
-- Ver los 10 usuarios más críticos con toda la información relevante
SELECT 
    id_usuario,
    nombre,
    apellido,
    segmento_riesgo,
    ROUND(probabilidad_fuga, 4) AS prob_fuga,
    segmento_valor,
    saldo,
    dias_desde_ultima_apuesta,
    deporte_favorito,
    nivel_frustracion,
    accion_recomendada,
    prioridad_contacto
FROM workspace.apuestas_db.vista_ml_churn_powerbi
WHERE segmento_riesgo IN ('Crítico', 'Alto Riesgo')
ORDER BY prioridad_contacto, probabilidad_fuga DESC
LIMIT 10;

id_usuario,nombre,apellido,segmento_riesgo,prob_fuga,segmento_valor,saldo,dias_desde_ultima_apuesta,deporte_favorito,nivel_frustracion,accion_recomendada,prioridad_contacto
237,René,Gálvez,Crítico,0.9996,Alto Valor,10473.29,83,Baloncesto,Frustración Moderada,Contacto Urgente + Bono Retención,2
352,Griselda,Quintana,Crítico,0.9996,Alto Valor,7250.76,91,Béisbol,Frustración Moderada,Contacto Urgente,2
270,Gabriela,Mercader,Crítico,0.9994,Alto Valor,24328.29,96,Tenis,Alta Frustración,Contacto Urgente + Bono Retención,2
218,Albano,Sanmartín,Crítico,0.9993,Alto Valor,10570.30,54,Tenis,Frustración Moderada,Contacto Urgente + Bono Retención,2
550,Amancio,Echevarría,Crítico,0.9993,Alto Valor,38024.07,132,Baloncesto,Alta Frustración,Contacto Urgente + Bono Retención,2
188,Anna,Suarez,Crítico,0.9993,Alto Valor,8568.89,129,Tenis,Alta Frustración,Contacto Urgente,2
389,Primitivo,Meléndez,Crítico,0.9993,Alto Valor,33914.06,116,Ciclismo,Alta Frustración,Contacto Urgente + Bono Retención,2
714,Blanca,Bilbao,Crítico,0.9993,Alto Valor,45335.24,100,Tenis,Alta Frustración,Contacto Urgente + Bono Retención,2
390,Joan,Alsina,Crítico,0.9992,Alto Valor,31318.98,41,Baloncesto,Alta Frustración,Contacto Urgente + Bono Retención,2
296,Tere,Benet,Crítico,0.9992,Alto Valor,35690.17,31,Béisbol,Alta Frustración,Contacto Urgente + Bono Retención,2


In [0]:
%sql
SELECT * FROM workspace.apuestas_db.vista_ml_churn_powerbi;

id_usuario,nombre,apellido,numero_documento,tipo_documento,fecha_registro,churn_real,probabilidad_fuga,segmento_riesgo,saldo,deporte_favorito,n_apuestas_pendientes,monto_apuestas_pendientes,dias_desde_ultima_apuesta,dias_desde_ultimo_deposito,antiguedad_dias,n_apuestas_total,n_apuestas_ult30d,n_apuestas_previas,tendencia_frecuencia,tendencia_actividad,inactivo_60_dias,sin_depositos_90_dias,veces_apostado_favorito,pct_apuestas_favorito,n_transacciones_deposito,n_transacciones_retiro,monto_total_depositado_tx,monto_total_retirado,pct_transacciones_deposito,patron_financiero,monto_promedio_apostado,monto_promedio_ult30d,monto_total_depositado,n_depositos,segmento_valor,n_apuestas_ganadas,n_apuestas_perdidas,n_apuestas_pendientes_desglose,tasa_perdida,nivel_frustracion,prioridad_contacto,ultima_apuesta_monto,ultima_apuesta_cuota,ultima_apuesta_fecha,ultima_apuesta_deporte,ultima_apuesta_estado,accion_recomendada,sin_apuestas_obs,sin_depositos_obs,sin_apuestas_resueltas
1,Trinidad,Pellicer,39264364,Cédula,2025-10-20T21:16:09.612Z,0,0.0,Estable,27864.03,Baloncesto,9,19411.82,3,12,243,31,11,20,-9,Estable,No,No,13,38.24,6,7,17782.43,21048.33,46.15,Más Retira que Deposita,2171.12,2634.85,11175.58,4,Alto Valor,6,19,9,0.7273,Frustración Moderada,6,2793.23,4.78,2026-07-16T03:45:52.877Z,Baloncesto,Perdida,Campaña Fidelización,0,0,0
2,Ezequiel,Murcia,93852080,Cédula,2026-01-05T01:17:31.316Z,1,0.9998135567166432,Crítico,37500.70,Baloncesto,0,0.00,109,112,166,1,0,1,-1,Estable,Sí,Sí,1,100.00,1,1,2667.63,2763.18,50.00,Balanceado,151.23,0.00,2667.63,1,Bajo Valor,0,1,0,1.0,Alta Frustración,3,151.23,3.67,2026-03-03T03:45:52.877Z,Baloncesto,Perdida,Contacto Urgente + Bono Retención,0,0,0
3,José Luis,Peñalver,66762856,Cédula,2025-08-23T16:25:52.396Z,1,0.9989641021983795,Crítico,23381.04,Béisbol,0,0.00,151,999,301,1,0,1,-1,Estable,Sí,Sí,1,100.00,0,2,0.00,5024.61,0.00,Más Retira que Deposita,1476.74,0.00,0.00,0,Bajo Valor,0,1,0,1.0,Alta Frustración,3,1476.74,8.66,2026-01-20T03:45:52.877Z,Béisbol,Perdida,Contacto Urgente + Bono Retención,0,1,0
4,Felisa,Quirós,33529348,DNI,2026-02-22T10:28:56.213Z,1,0.9985430441455911,Crítico,6124.71,Ciclismo,1,392.03,31,62,118,4,0,4,-4,Estable,No,No,2,50.00,2,0,906.55,0.00,100.00,Solo Deposita,711.42,0.00,906.55,2,Bajo Valor,1,2,1,0.6667,Frustración Moderada,3,392.03,2.86,2026-05-20T03:45:52.877Z,Baloncesto,Pendiente,Contacto Urgente,0,0,0
5,Rosalinda,Cózar,22019027,Cédula,2026-05-05T04:05:44.707Z,0,0.004328767123287672,Estable,7789.49,Baloncesto,13,33770.04,5,31,46,27,5,22,-17,Descendente,No,No,11,34.38,2,8,5311.59,24101.03,20.00,Más Retira que Deposita,2451.73,2239.50,5311.59,2,Medio Valor,10,9,13,0.5625,Normal,6,2714.08,5.83,2026-07-07T03:45:52.877Z,Fútbol,Ganada,Campaña Fidelización,0,0,0
6,Pancho,Capdevila,96451170,Pasaporte,2026-01-17T02:44:21.900Z,0,0.0,Estable,20269.43,Baloncesto,10,35027.93,5,40,154,21,4,17,-13,Descendente,No,No,10,35.71,6,4,18890.77,17145.68,60.00,Más Deposita que Retira,2761.37,2981.74,12244.36,4,Alto Valor,5,13,10,0.6667,Frustración Moderada,6,1464.13,4.33,2026-07-15T03:45:52.877Z,Fútbol,Perdida,Campaña Fidelización,0,0,0
7,Laura,Contreras,68562177,Pasaporte,2026-04-19T10:48:19.186Z,0,0.0,Estable,29356.97,Baloncesto,11,30090.05,6,86,62,21,1,20,-19,Descendente,No,No,7,31.82,4,6,7753.81,14586.05,40.00,Más Retira que Deposita,2747.39,3808.25,7753.81,4,Medio Valor,2,9,11,0.9,Alta Frustración,6,2006.96,4.18,2026-07-18T03:45:52.877Z,Tenis,Ganada,Campaña Fidelización,0,0,0
8,Amado,Tomás,99893307,DNI,2026-03-20T04:07:27.784Z,0,0.005,Estable,39052.97,Béisbol,7,24264.99,3,22,92,20,6,14,-8,Estable,No,No,7,26.92,6,6,17955.33,20854.12,50.00,Balanceado,2471.81,1175.93,11406.41,4,Alto Valor,6,13,7,0.7143,Frustración Moderada,6,4493.37,4.60,2026-07-16T03:45:52.877Z,Béisbol,Pendiente,Campaña Fidelización,0,0,0
9,Ruy,Narváez,35784031,Pasaporte,2025-07-23T02:07:10.369Z,0,0.0,Estable,16572.63,Baloncesto,7,20256.51,6,7,332,18,5,13,-8,Estable,No,No,7,31.82,7,5,18677.60,12329.50,58.33,Más Deposita que Retira,2217